In [12]:
# @title Challenge 4: Workflow Agents & Setup
from google.adk.agents import Agent, SequentialAgent, LoopAgent
from google.adk.tools.google_search_tool import GoogleSearchTool
from google.adk.tools import agent_tool

MODEL_GEMINI = "gemini-2.5-flash"

print("Workflow modules imported successfully.")

Workflow modules imported successfully.


In [17]:
# @title Tool Definition: URL Link Validator
import requests
from typing import Dict

def validate_recipe_url(url: str) -> Dict[str, str]:
    """
    Checks if a given recipe URL is accessible and returns a 200 OK status.
    """
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        response = requests.head(url, headers=headers, timeout=5, allow_redirects=True)

        # Fallback to GET if HEAD is blocked by the server
        if response.status_code >= 400:
            response = requests.get(url, headers=headers, timeout=5)

        if response.status_code == 200:
            return {"url": url, "status": "verified", "message": "Link is active and accessible."}
        else:
            return {"url": url, "status": "broken", "message": f"Link returned status code {response.status_code}."}
    except Exception as e:
        return {"url": url, "status": "error", "message": str(e)}

# Quick test on your fixed links
print(validate_recipe_url("https://juliasalbum.com/chicken-stir-fry/"))
print(validate_recipe_url("https://oliviaadriance.com/roasted-lemon-chicken-and-potatoes/"))

{'url': 'https://juliasalbum.com/chicken-stir-fry/', 'status': 'verified', 'message': 'Link is active and accessible.'}
{'url': 'https://oliviaadriance.com/roasted-lemon-chicken-and-potatoes/', 'status': 'verified', 'message': 'Link is active and accessible.'}


In [20]:
# @title Define Search, Critique, and Refine Agents for Kitchen-Pal

# 1. Search Agent: Finds recipe data and checks constraints and include the validation function as a tool
search_agent = Agent(
    name="kitchen_search_agent",
    model=MODEL_GEMINI,
    description="Searches the web for recipe candidates matching dietary restrictions, inventory items, and budget constraints while verifying link accessibility.",
    instruction="""You are a culinary research assistant. When given meal constraints (such as budget, inventory items, and dietary exclusions):
1. Search the web using Google Search to find viable recipes.
2. CRITICAL: Every single recipe found MUST include its exact, direct source URL. Never omit source links.
3. Use the validate_recipe_url tool to ensure the links are active and accessible.""",
    tools=[GoogleSearchTool(), validate_recipe_url]
)

# 2. Critique Agent: Reviews the meal plan against budget, time limits, and dietary rules
critique_agent = Agent(
    name="kitchen_critique_agent",
    model=MODEL_GEMINI,
    description="Evaluates the proposed meal plan against budget limits, time caps, and dietary rules.",
    instruction="""You are a strict culinary critic and budget auditor. Review the proposed meal plan against the user's constraints:
- Check if total cost is under the budget limit.
- Ensure pork is strictly avoided.
- Verify that at least two meals are under 30 minutes.
- Check if inventory items (chicken thighs, spinach) are utilized.
If any constraints fail, provide explicit suggestions for replacement or cost reduction. If everything satisfies the criteria, state 'APPROVED'."""
)

# 3. Refine Agent: Rewrites and formats the final structured plan
refine_agent = Agent(
    name="kitchen_refine_agent",
    model=MODEL_GEMINI,
    description="Refines and formats the meal plan based on critique feedback.",
    instruction="""You are a professional recipe formatter and meal-planning assistant. Take the researched recipes and the critic's feedback, and output a clean, final response containing:
1. Selected meal plan
2. Importable recipe records featuring **direct, clickable markdown hyperlinks** for every source URL (e.g., [Recipe Title](URL))
3. Consolidated shopping list
4. Cost and serving analysis
Never drop or omit the source URLs provided by the search agent."""
)

print("Specialized workflow agents defined.")

Specialized workflow agents defined.


In [14]:
# @title Configure the Corrected Sequential Agent Workflow Team
from google.adk.agents import SequentialAgent

# Order: Search finds data -> Critique evaluates/audits -> Refine formats the final output
kitchen_refinement_room = SequentialAgent(
    name="kitchen_refinement_room",
    description="Executes a sequential culinary pipeline: researches, critiques, and finalizes the meal plan.",
    sub_agents=[
        search_agent,
        critique_agent,  # 1. Critique runs first to check requirements
        refine_agent     # 2. Refine runs last to write out the final clean output
    ]
)

print("SequentialAgent updated with Refine as the final output agent.")

SequentialAgent updated with Refine as the final output agent.


/tmp/ipykernel_58646/375347767.py:5: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  kitchen_refinement_room = SequentialAgent(


In [15]:
# @title Define Greeter Root Agent with State Statefulness
def append_to_state(tool_context, field, response):
    existing_state = tool_context.state.get(field, [])
    tool_context.state[field] = existing_state + [response]
    return {"status": "success"}

GREETER_INSTRUCTIONS = """
You are Kitchen-Pal's primary Greeter and Orchestrator.
Receive the user's meal planning request, initialize the session context, and pass the prompt to the kitchen_refinement_room workflow team to generate the verified, budget-checked meal plan.
"""

greeter_agent = Agent(
    name="greeter",
    model=MODEL_GEMINI,
    description="Primary entry point for Kitchen-Pal user requests.",
    instruction=GREETER_INSTRUCTIONS,
    tools=[append_to_state],
    sub_agents=[kitchen_refinement_room],
)

print("Greeter root agent configured.")

Greeter root agent configured.


In [16]:
# @title Test Execution: Running the Challenge Four Workflow
from IPython.display import Markdown, display
from vertexai.preview import reasoning_engines

# Initialize app host with the greeter root agent
app_workflow = reasoning_engines.AdkApp(agent=greeter_agent)

user_id = "kitchen-pal-user"
session = app_workflow.create_session(user_id=user_id)
session_id = session["id"]

user_request = (
    "Plan five dinners for two adults and one child. Stay under $80, avoid pork, "
    "include two meals under 30 minutes, use chicken thighs and spinach from my inventory, "
    "and provide importable recipe links and a consolidated shopping list."
)

print(f"User Request:\n{user_request}\n")
print("=" * 60)
print("Running multi-agent refinement workflow (Search -> Refine -> Critique)...")
print("=" * 60)

last_event = None
for event in app_workflow.stream_query(
    user_id=user_id,
    session_id=session_id,
    message=user_request
):
    last_event = event

if last_event and "content" in last_event and "parts" in last_event["content"]:
    response_text = last_event["content"]["parts"][0]["text"]
    display(Markdown(f"**Kitchen-Pal Final Verified Output:**\n\n{response_text}"))
else:
    print("No response received from workflow execution.")

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:966: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


User Request:
Plan five dinners for two adults and one child. Stay under $80, avoid pork, include two meals under 30 minutes, use chicken thighs and spinach from my inventory, and provide importable recipe links and a consolidated shopping list.

Running multi-agent refinement workflow (Search -> Refine -> Critique)...


**Kitchen-Pal Final Verified Output:**

Here is your refined 5-day dinner plan, tailored for two adults and one child, staying under budget, avoiding pork, including quick meals, and utilizing your inventory items. All criteria have been met and approved.

---

### **Selected Meal Plan**

**Meal 1: Mediterranean Chicken Stir Fry with Spinach and Feta**
*   **Description:** This vibrant one-pan stir-fry utilizes your chicken thighs and spinach, combined with fresh tomatoes and salty feta, for a quick and flavorful meal.
*   **Estimated Time:** 30 minutes.
*   **Serving Size:** 2-3 servings (can be stretched with a side of rice).
*   **Notes:** This recipe uses skinless boneless chicken thighs, but your bone-in chicken thighs can be deboned and cut into pieces, or you can adjust cooking time if using bone-in. Ensure spinach from your inventory is used.

**Meal 2: Creamy Tomato Spinach Pasta**
*   **Description:** A rich and comforting pasta dish with a creamy tomato sauce and plenty of fresh spinach. It's a perfect quick weeknight meal.
*   **Estimated Time:** Under 30 minutes.
*   **Serving Size:** 3-4 servings.
*   **Notes:** This recipe is vegetarian, making it budget-friendly. It's an excellent way to use your spinach inventory.

**Meal 3: One-Pan Roasted Lemon Herb Chicken and Potatoes**
*   **Description:** Juicy roasted chicken thighs with tender potatoes, infused with lemon and herbs, all cooked on a single pan for easy cleanup.
*   **Estimated Time:** 75-80 minutes (60 min bake time + prep).
*   **Serving Size:** 3-4 servings.
*   **Notes:** This recipe is ideal for using your bone-in, skin-on chicken thighs.

**Meal 4: Easy Ground Beef Tacos**
*   **Description:** A family-friendly and highly customizable meal, perfect for a quick and fun dinner.
*   **Estimated Time:** Less than 20 minutes.
*   **Serving Size:** 4-6 servings.
*   **Notes:** Customize toppings to your family's preferences. Use pre-made taco seasoning for extra speed, or mix your own with common pantry spices.

**Meal 5: Hearty Lentil Soup**
*   **Description:** A warming and nutritious lentil soup, packed with vegetables, that's incredibly budget-friendly and yields comforting leftovers.
*   **Estimated Time:** 45-60 minutes cooking time + prep.
*   **Serving Size:** 6-8 servings (provides excellent leftovers).
*   **Notes:** This recipe is naturally vegetarian and very economical.

---

### **Importable Recipe Records**

*   **Mediterranean Chicken Stir Fry with Spinach and Feta:** [https://juliasalbum.com/mediterranean-chicken-stir-fry-with-vegetables/](https://juliasalbum.com/mediterranean-chicken-stir-fry-with-vegetables/)
*   **Creamy Tomato Spinach Pasta:** [https://www.makingthymeforhealth.com/creamy-tomato-spinach-pasta/](https://www.makingthymeforhealth.com/creamy-tomato-spinach-pasta/)
*   **One-Pan Roasted Lemon Herb Chicken and Potatoes:** [https://oliviaadriance.com/one-pan-roasted-lemon-herb-chicken-and-potatoes/](https://oliviaadriance.com/one-pan-roasted-lemon-herb-chicken-and-potatoes/)
*   **Easy Ground Beef Tacos:** [https://mountainmamacooks.com/easy-ground-beef-tacos/](https://mountainmamacooks.com/easy-ground-beef-tacos/)
*   **Hearty Lentil Soup:** [https://www.elizabethrider.com/easy-lentil-soup-recipe/](https://www.elizabethrider.com/easy-lentil-soup-recipe/)

---

### **Consolidated Shopping List**

**Produce:**
*   Lemons (4-5)
*   Baby Potatoes (2 lbs)
*   Yellow Onions (2 medium)
*   Garlic (1 head)
*   Cherry Tomatoes (1 pint)
*   Carrots (3-4 medium)
*   Celery (3-4 stalks)
*   Lettuce (1 head or bag, for tacos)
*   Fresh Cilantro (1 bunch, optional for tacos)
*   Avocado (1-2, optional for tacos)
*   Fresh Rosemary (1 small bunch, or dried equivalent)
*   Arugula (1 bag, optional for roasted chicken)

**Pantry:**
*   Penne Pasta (1 lb)
*   Crushed Tomatoes (1 large can, ~28 oz)
*   Vegetable or Chicken Broth (approx. 6-8 cups total)
*   Dried Lentils (1.5 cups)
*   Olive Oil (check inventory to ensure enough for all recipes)
*   Spices: Smoked Paprika, Dried Oregano, Dried Basil, Chili Powder, Ground Cumin, Salt, Black Pepper (check your pantry for these common spices; consider a taco seasoning packet for convenience)
*   Diced Tomatoes (1 can, 14.5 oz - *alternative to fresh cherry tomatoes if preferred for stir-fry*)

**Dairy/Meat/Other:**
*   Ground Beef (1 lb, 80/20 recommended for flavor)
*   Cream Cheese (8 oz, regular or dairy-free)
*   Feta Cheese (4 oz, crumbled or block)
*   Shredded Cheese (e.g., Cheddar or Mexican blend, 8 oz, for tacos)
*   Taco Shells or Tortillas (1 pack)
*   Sour Cream (1 small container, optional for tacos)

**Inventory Items (Already on Hand):**
*   Chicken Thighs (sufficient for 2 meals)
*   Spinach (sufficient for 2 meals)

---

### **Cost and Serving Analysis**

*   **Estimated Total Shopping Cost:** Approximately $59.50 (This estimate is well within your $80 budget, but actual prices may vary based on your local grocery store).
*   **Total Servings:** This plan provides approximately 18-24 servings across five dinners, including potential leftovers, especially from the Lentil Soup and Tacos, which is generous for two adults and one child.

In [ ]:
# @title Interactive Kitchen-Pal Chat Session
from IPython.display import Markdown, display
from vertexai.preview import reasoning_engines

# 1. Initialize the app session with your greeter root agent
interactive_app = reasoning_engines.AdkApp(agent=greeter_agent)
chat_user_id = "kitchen-pal-chat-user"
chat_session = interactive_app.create_session(user_id=chat_user_id)
chat_session_id = chat_session["id"]

EXIT_PHRASES = {"exit", "quit", "bye", "goodbye", "done", "stop"}

print("==================================================")
print("  Kitchen-Pal Interactive Assistant Initialized!")
print("  Type your meal planning requests below.")
print("  Type 'exit' or 'quit' to end the chat session.")
print("==================================================\n")

while True:
    user_prompt = input("You: ").strip()
    if not user_prompt:
        continue

    if user_prompt.lower() in EXIT_PHRASES:
        print("\nKitchen-Pal session ended. Happy cooking!\n")
        break

    print("\nKitchen-Pal is coordinating the search, critique, and refinement workflow...\n")

    try:
        last_event = None
        for event in interactive_app.stream_query(
            user_id=chat_user_id,
            session_id=chat_session_id,
            message=user_prompt
        ):
            last_event = event

        if last_event and "content" in last_event and "parts" in last_event["content"]:
            response_text = last_event["content"]["parts"][0]["text"]
            display(Markdown(f"**Kitchen-Pal:**\n\n{response_text}"))
        else:
            print("Kitchen-Pal: [No response received]")

    except Exception as e:
        print(f"\nAn error occurred during workflow execution: {e}")

    print("\n" + "-" * 50 + "\n")

  Kitchen-Pal Interactive Assistant Initialized!
  Type your meal planning requests below.
  Type 'exit' or 'quit' to end the chat session.

You: Plan three quick dinners for two adults under $50 using ground beef and spinach, avoid pork, and include verified recipe links and a shopping list.

Kitchen-Pal is coordinating the search, critique, and refinement workflow...



**Kitchen-Pal:**

Here is your refined meal plan, addressing all feedback from the `kitchen_critique_agent`. The plan now ensures all criteria are met, including budget, ingredient utilization, and quick preparation times.

---

### **Selected Meal Plan**

This meal plan offers three quick, budget-friendly dinners for two adults, utilizing ground beef, spinach, and chicken thighs (as per inventory assumption in critique), while strictly avoiding pork.

1.  **Dinner 1: Hamburger Skillet with Mushrooms and Spinach**
2.  **Dinner 2: Creamy Ground Beef Pasta with Spinach & Parmesan (One-Pot)**
3.  **Dinner 3: Creamy Lemon Herb Chicken & Spinach** (Replaced "Weeknight Beef Stuffed Spinach Peppers" to utilize chicken thighs and optimize cost/time)

---

### **Importable Recipe Records**

---

#### **Recipe 1: Hamburger Skillet with Mushrooms and Spinach**

*   **Source:** [Hamburger Skillet with Mushrooms and Spinach - The Buttered Home](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH8FHoyylB7AVuWxyTXVj1C3ccDPr5bPXND21GYjTKYjpWD3fWXEKAKYJnwaEl7TFr9ituuxymfa9jrjjhrZ45DpY9211pIaRlcpYmc9EZSMa_K3808LFmltsdSUUOO_7vWEcWi51n8Aa_LR9aEH4vBl0Mx7weyoW4_86gn82ElMzb_NMI=)
*   **Estimated Time:** 20-25 minutes prep and cook time.
*   **Serving Size:** 2 adults.
*   **Ingredients:**
    *   0.75 lb lean ground beef
    *   4 oz sliced mushrooms
    *   1.5 cups fresh baby spinach
    *   0.25 cup chopped yellow onion
    *   1 tbsp minced garlic
    *   3 large eggs
    *   0.25 tsp salt
    *   0.25 tsp black pepper
    *   0.25 cup grated Parmesan cheese
    *   1 tbsp olive oil or butter

---

#### **Recipe 2: Creamy Ground Beef Pasta with Spinach & Parmesan (One-Pot)**

*   **Source:** [Creamy Ground Beef Pasta with Spinach & Parmesan | Easy One Pot Dinner Recipe](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHbTLXxV6IxFSLQbKqCAjGN-e0Qn43cx7cX0YcgSb_WuExQ_hqYwwYrRw6sIGYF0kMBMyPCt4NMMRCpkD-enr8A89ggKnexq9lHo4K-m_M-o4J8wjscBjwiDIHJaBK7cxKC0SiDWQQ=)
*   **Estimated Time:** 25-30 minutes prep and cook time.
*   **Serving Size:** 2 adults.
*   **Ingredients:**
    *   0.5 lb ground beef
    *   6-8 oz pasta (e.g., penne, rotini), about half a standard box
    *   2 cups fresh spinach
    *   0.5 medium yellow onion, chopped
    *   1-2 cloves garlic, minced
    *   1 tbsp tomato paste
    *   0.25 cup heavy cream
    *   2 tbsp grated Parmesan cheese
    *   1 tbsp olive oil
    *   Salt, black pepper, paprika, oregano, chili flakes (to taste)

---

#### **Recipe 3: Creamy Lemon Herb Chicken & Spinach**

*   **Source:** *Conceptual recipe based on `kitchen_critique_agent` suggestion for inventory utilization.*
*   **Estimated Time:** 25-30 minutes prep and cook time.
*   **Serving Size:** 2 adults.
*   **Ingredients:**
    *   1 lb boneless, skinless chicken thighs, cut into 1-inch pieces
    *   5 oz fresh baby spinach
    *   0.25 cup heavy cream
    *   2 cloves garlic, minced
    *   1 tbsp olive oil
    *   0.5 lemon, juiced
    *   1 tsp dried Italian herbs
    *   Salt, black pepper (to taste)
*   **Instructions:**
    1.  Heat olive oil in a large skillet over medium-high heat. Season chicken thigh pieces with salt, pepper, and dried Italian herbs.
    2.  Add seasoned chicken to the hot skillet and cook, stirring occasionally, until browned and cooked through (internal temperature 165°F/74°C), about 5-7 minutes. Remove chicken from skillet and set aside.
    3.  Add minced garlic to the skillet and cook for 30 seconds until fragrant.
    4.  Pour in heavy cream and lemon juice, stirring well. Bring to a gentle simmer.
    5.  Add the fresh spinach to the skillet, stirring until it wilts completely, about 2-3 minutes.
    6.  Return the cooked chicken to the skillet, tossing gently to coat in the creamy sauce. Taste and adjust seasoning if needed. Serve immediately.

---

### **Consolidated Shopping List**

*(This list is optimized for minimal waste and cost, assuming common pantry staples are already on hand.)*

**Proteins & Dairy:**
*   **Ground Beef:** 1.25 lbs (to cover 0.75 lb for D1 and 0.5 lb for D2)
*   **Boneless, Skinless Chicken Thighs:** 1 lb
*   **Large Eggs:** 3
*   **Heavy Cream:** 0.5 cup (purchase a small carton, e.g., 1 pint/2 cups, and use what's needed)
*   **Parmesan Cheese:** ~0.4 cup total (purchase a small container of grated or shredded)

**Vegetables & Produce:**
*   **Fresh Baby Spinach:** 1 x 10 oz bag (sufficient for 8.5 oz total usage across all meals)
*   **Mushrooms:** 4 oz (pre-sliced or whole)
*   **Yellow Onions:** 1-2 medium
*   **Garlic:** 1 head
*   **Lemon:** 1 (you'll use half)

**Pantry Items:**
*   **Pasta:** 8 oz (e.g., penne, rotini; half a 1 lb box)
*   **Tomato Paste:** 1 small can or tube (1 tbsp needed)

**Assumed Pantry Staples (check if you have these):**
*   Olive oil or other cooking oil
*   Salt
*   Black pepper
*   Paprika
*   Oregano
*   Chili flakes
*   Dried Italian herbs

---

### **Cost and Serving Analysis**

*   **Total Estimated Cost:** **$26 - $40** (This estimate is significantly reduced and well within the $50 budget, addressing the previous critique.)
    *   *Detailed breakdown reflects purchasing necessary quantities of main ingredients and the smallest available sizes for shared ingredients like cream/Parmesan/pasta.*
    *   *Actual cost may vary by location, store sales, and specific brands chosen.*
*   **Servings:** Each recipe serves 2 adults, providing three dinners for a total of 6 servings.
*   **Quick Preparation:**
    *   Dinner 1: 20-25 minutes
    *   Dinner 2: 25-30 minutes
    *   Dinner 3: 25-30 minutes
    *   All three meals are completed within or just at the 30-minute mark, fulfilling the "quick dinners" requirement.
*   **Ingredient Utilization:**
    *   **Ground Beef:** Effectively utilized across two meals.
    *   **Spinach:** Precisely accounted for, minimizing waste by suggesting one 10oz bag for approximately 8.5oz needed.
    *   **Chicken Thighs:** Fully utilized in the new Dinner 3, addressing the inventory requirement from the critique.
    *   **Pork:** Strictly avoided.


--------------------------------------------------

